# Train — DBpedia (in-batch negatives)

**Purpose:** Trains the Hybrid SPARQL encoder on DBpedia semantic triplets using **in-batch negative** training: one positive per anchor, with other batch items as negatives (InfoNCE). No explicit negative graphs are built; the batch itself provides negatives.

**Input:** `../LSQ_queries/semantic_triplets/semantic_triplets_dbpedia.json` (from `02_dbpedia_semantic_triplets.ipynb`).

**Output:**
- `splits/semantic_triplet_splits_dbpedia.json` — train/valid/test indices (created on first run).
- `../checkpoints/dbpedia_model_semantic-20.pt` — trained model weights.

**Config:** See `src.config.Config` (device, epochs, learning rate, temperature). Batch size for in-batch training is 16.

**Sections:** 1. Imports and data load | 2. Splits | 3. Dataset and DataLoaders (in-batch) | 4. Loss | 5. Training and save.


### 1. Imports and data load

In [1]:
import sys, os
if "src" not in sys.path:
    sys.path.append(os.path.abspath(".."))

os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")

import json, random, numpy as np, torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset

from src.config import Config
from src.graph_builder import SPARQLGraphBuilder
from src.model import HybridSPARQLEncoder
from src.semantic_dataset import SemanticSPARQLDataset, semantic_collate_fn_inbatch

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

SEMANTIC_PATH = "../LSQ_queries/semantic_triplets/semantic_triplets_dbpedia.json"

with open(SEMANTIC_PATH, "r", encoding="utf-8") as f:
    raw_sem = json.load(f)

print(f"Total semantic entries: {len(raw_sem)}")

/upb/users/g/ganeshk/profiles/unix/cs/.conda/envs/sparql-emb/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Total semantic entries: 4713


### 2. Splits

In [2]:
import os

SPLITS_PATH = "splits/semantic_triplet_splits_dbpedia.json"

if os.path.exists(SPLITS_PATH):
    with open(SPLITS_PATH, "r", encoding="utf-8") as f:
        split_info = json.load(f)
    train_idx = split_info["train_idx"]
    valid_idx = split_info["valid_idx"]
    test_idx = split_info["test_idx"]
    print(f"Loaded existing splits from {SPLITS_PATH}")
    print(f"Train: {len(train_idx)}, Valid: {len(valid_idx)}, Test: {len(test_idx)}")
else:
    indices = list(range(len(raw_sem)))
    random.shuffle(indices)

    n = len(indices)
    train_size = int(0.8 * n)
    valid_size = int(0.1 * n)
    test_size  = n - train_size - valid_size

    train_idx = indices[:train_size]
    valid_idx = indices[train_size : train_size + valid_size]
    test_idx  = indices[train_size + valid_size :]

    print(f"Train: {len(train_idx)}, Valid: {len(valid_idx)}, Test: {len(test_idx)}")

    os.makedirs("splits", exist_ok=True)
    split_info = {
        "seed": SEED,
        "train_idx": train_idx,
        "valid_idx": valid_idx,
        "test_idx": test_idx,
    }
    with open(SPLITS_PATH, "w", encoding="utf-8") as f:
        json.dump(split_info, f, indent=2)

    print(f"Saved splits to {SPLITS_PATH}")


Train: 3770, Valid: 471, Test: 472
Saved splits to splits/semantic_triplet_splits_dbpedia.json


### Dataset and DataLoaders (in-batch)

In [3]:
print("Initializing semantic dataset...")
builder = SPARQLGraphBuilder()
full_sem = SemanticSPARQLDataset(SEMANTIC_PATH, builder, in_batch_negatives=True)
BATCH_SIZE_TRAIN = 16

print(f"Total semantic anchors: {len(full_sem)}")

train_ds = Subset(full_sem, train_idx)
valid_ds = Subset(full_sem, valid_idx)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE_TRAIN,
    collate_fn=semantic_collate_fn_inbatch,
    shuffle=True,
    num_workers=0,
)
valid_loader = DataLoader(
    valid_ds,
    batch_size=BATCH_SIZE_TRAIN,
    collate_fn=semantic_collate_fn_inbatch,
    shuffle=False,
    num_workers=0,
)

print(f"Train: {len(train_ds)} | Valid: {len(valid_ds)}")


Initializing semantic dataset...
Loading semantic dataset from ../LSQ_queries/semantic_triplets/semantic_triplets_dbpedia.json...
Generated 4713 semantic anchor groups.
Total semantic anchors: 4713
Train: 3770 | Valid: 471


### Loss (in-batch InfoNCE)

In [4]:
import torch


def infonce_inbatch(anchor_emb, pos_emb, temperature: float):
    anchor_emb = F.normalize(anchor_emb, dim=1)
    pos_emb = F.normalize(pos_emb, dim=1)
    logits = (anchor_emb @ pos_emb.T) / temperature
    labels = torch.arange(logits.size(0), device=logits.device)
    return F.cross_entropy(logits, labels)


### Training loop and save

In [5]:
model = HybridSPARQLEncoder().to(Config.DEVICE)
model.train()
optimizer = torch.optim.AdamW(model.parameters(), lr=Config.LEARNING_RATE)

print(f"Training on {Config.DEVICE} with {len(train_ds)} samples...")

for epoch in range(Config.EPOCHS):
    total_loss = 0.0

    for anchor, pos, pos_js in train_loader:
        anchor = anchor.to(Config.DEVICE)
        pos = pos.to(Config.DEVICE)
        pos_js = pos_js.to(Config.DEVICE)

        optimizer.zero_grad()
        a_emb = model(anchor)
        p_emb = model(pos)
        loss_ce = infonce_inbatch(a_emb, p_emb, Config.TEMPERATURE)
        loss_reg = ((a_emb * p_emb).sum(dim=1) - pos_js).pow(2).mean()
        loss = loss_ce + 0.3 * loss_reg

        loss.backward()
        optimizer.step()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        total_loss += loss.item()

    avg_train_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{Config.EPOCHS} | Train Loss: {avg_train_loss:.4f}")

    model.eval()
    with torch.no_grad():
        sims_pos, count = 0.0, 0
        for batch in valid_loader:
            anchor, pos, pos_js = batch
            anchor = anchor.to(Config.DEVICE)
            pos = pos.to(Config.DEVICE)
            a_emb = F.normalize(model(anchor), dim=1)
            p_emb = F.normalize(model(pos), dim=1)
            sims_pos += (a_emb * p_emb).sum(dim=1).sum().item()
            count += anchor.size(0)
        print(f"          Valid sim(anchor,pos): {sims_pos / count:.4f}")

    model.train()


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 482.25it/s, Materializing param=pooler.dense.weight]                               


Training on cuda with 3770 samples...
Epoch 1/20 | Train Loss: 1.8111
          Valid sim(anchor,pos): 0.2737
Epoch 2/20 | Train Loss: 1.2555
          Valid sim(anchor,pos): 0.2708
Epoch 3/20 | Train Loss: 1.1698
          Valid sim(anchor,pos): 0.2694
Epoch 4/20 | Train Loss: 1.1062
          Valid sim(anchor,pos): 0.2678
Epoch 5/20 | Train Loss: 1.0456
          Valid sim(anchor,pos): 0.2698
Epoch 6/20 | Train Loss: 0.9848
          Valid sim(anchor,pos): 0.2670
Epoch 7/20 | Train Loss: 0.9267
          Valid sim(anchor,pos): 0.2671
Epoch 8/20 | Train Loss: 0.8990
          Valid sim(anchor,pos): 0.2672
Epoch 9/20 | Train Loss: 0.8547
          Valid sim(anchor,pos): 0.2668
Epoch 10/20 | Train Loss: 0.8165
          Valid sim(anchor,pos): 0.2645
Epoch 11/20 | Train Loss: 0.7904
          Valid sim(anchor,pos): 0.2630
Epoch 12/20 | Train Loss: 0.7577
          Valid sim(anchor,pos): 0.2606
Epoch 13/20 | Train Loss: 0.6872
          Valid sim(anchor,pos): 0.2603
Epoch 14/20 | Train Lo

In [6]:
os.makedirs("../checkpoints", exist_ok=True)
torch.save(model.state_dict(), "../checkpoints/dbpedia_model_semantic-20.pt")
print("Semantic training complete. Model saved to ../checkpoints/dbpedia_model_semantic-20.pt")


Semantic training complete. Model saved to ../checkpoints/dbpedia_model_semantic-20.pt
